In [1]:
# Import necessary libraries
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split  # Import train_test_split to fix NameError
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from category_encoders.leave_one_out import LeaveOneOutEncoder
import numpy as np

In [ ]:
df = pd.read_csv("crop_yield_dataset.csv")

In [8]:
df

,Temperature (°C),Rainfall (mm),Humidity (%),Soil Type,Weather Condition,Crop Type,Yield (tons/hectare)
0,22.490802,185.132929,53.085284,Sandy,Sunny,Barley,2.818937
1,34.014286,541.900947,52.348940,Loamy,Sunny,Corn,8.014166
2,29.639879,872.945836,85.312729,Peaty,Rainy,Wheat,9.249868
3,26.973170,732.224886,52.477310,Sandy,Sunny,Soybeans,7.947481
4,18.120373,806.561148,53.597486,Clay,Stormy,Barley,6.262616
...,...,...,...,...,...,...,...
995,16.831641,656.955156,83.264788,Sandy,Sunny,Corn,6.822771
996,33.346272,956.614621,47.863660,Peaty,Rainy,Soybeans,7.820291
997,17.736373,68.958016,55.489393,Loamy,Sunny,Corn,2.921246
998,34.004747,57.054721,54.502277,Loamy,Rainy,Rice,6.705009


In [7]:

df.columns


Index(['Temperature (°C)', 'Rainfall (mm)', 'Humidity (%)', 'Soil Type',
       'Weather Condition', 'Crop Type', 'Yield (tons/hectare)'],
      dtype='object')

In [9]:
df = df.rename(columns={
    "Temperature (°C)": "temperature",
    "Rainfall (mm)": "rainfall",
    "Humidity (%)": "humidity",
    "Soil Type": "soil_type",
    "Weather Condition": "weather_condition",
    "Crop Type": "crop_type",
    "Yield (tons/hectare)": "yield"
})

In [10]:
df.columns

Index(['temperature', 'rainfall', 'humidity', 'soil_type', 'weather_condition',
       'crop_type', 'yield'],
      dtype='object')

In [11]:
 #Split data
X = df.drop("yield", axis=1)
y = df["yield"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
X_train

,temperature,rainfall,humidity,soil_type,weather_condition,crop_type
29,15.929008,828.915474,72.708730,Peaty,Sunny,Wheat
535,33.895315,121.886093,81.707230,Peaty,Rainy,Rice
695,27.212401,68.172309,65.951614,Clay,Stormy,Wheat
557,15.575654,411.028470,54.978284,Sandy,Stormy,Wheat
836,33.296918,128.095746,89.662751,Loamy,Cloudy,Corn
...,...,...,...,...,...,...
106,23.207658,396.242019,82.858963,Sandy,Rainy,Corn
270,31.187223,694.981886,45.211240,Clay,Stormy,Corn
860,30.510552,776.447446,83.338518,Clay,Cloudy,Barley
435,29.019383,29.973590,59.474095,Clay,Sunny,Barley


In [14]:
# OneHotEncoding
X_train_ohe = pd.get_dummies(X_train, columns=["soil_type", "weather_condition", "crop_type"], drop_first=True)
X_test_ohe = pd.get_dummies(X_test, columns=["soil_type", "weather_condition", "crop_type"], drop_first=True)

In [15]:
X_train_ohe

,temperature,rainfall,humidity,soil_type_Loamy,soil_type_Peaty,soil_type_Sandy,soil_type_Silty,weather_condition_Rainy,weather_condition_Stormy,weather_condition_Sunny,crop_type_Corn,crop_type_Rice,crop_type_Soybeans,crop_type_Wheat
29,15.929008,828.915474,72.708730,False,True,False,False,False,False,True,False,False,False,True
535,33.895315,121.886093,81.707230,False,True,False,False,True,False,False,False,True,False,False
695,27.212401,68.172309,65.951614,False,False,False,False,False,True,False,False,False,False,True
557,15.575654,411.028470,54.978284,False,False,True,False,False,True,False,False,False,False,True
836,33.296918,128.095746,89.662751,True,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,23.207658,396.242019,82.858963,False,False,True,False,True,False,False,True,False,False,False
270,31.187223,694.981886,45.211240,False,False,False,False,False,True,False,True,False,False,False
860,30.510552,776.447446,83.338518,False,False,False,False,False,False,False,False,False,False,False
435,29.019383,29.973590,59.474095,False,False,False,False,False,False,True,False,False,False,False


In [16]:
# Initialize or load LeaveOneOutEncoder
try:
    with open("looe_encoder.pkl", "rb") as f:
        looe_encoder = pickle.load(f)
    print("Loaded existing looe_encoder.pkl")
except FileNotFoundError:
    print("looe_encoder.pkl not found. Creating and fitting a new LeaveOneOutEncoder.")
    looe_encoder = LeaveOneOutEncoder(cols=["soil_type", "weather_condition", "crop_type"])
    looe_encoder.fit(X_train, y_train)
    with open("looe_encoder.pkl", "wb") as f:
        pickle.dump(looe_encoder, f)

Loaded existing looe_encoder.pkl


In [17]:
# Apply LeaveOneOutEncoder
X_train_looe = looe_encoder.transform(X_train)
X_test_looe = looe_encoder.transform(X_test)

X_train_looe

,temperature,rainfall,humidity,soil_type,weather_condition,crop_type
29,15.929008,828.915474,72.708730,5.956709,5.810522,6.153961
535,33.895315,121.886093,81.707230,5.956709,5.777420,5.917129
695,27.212401,68.172309,65.951614,5.662720,5.912701,6.153961
557,15.575654,411.028470,54.978284,6.006902,5.912701,6.153961
836,33.296918,128.095746,89.662751,5.989928,6.100385,5.632905
...,...,...,...,...,...,...
106,23.207658,396.242019,82.858963,6.006902,5.777420,5.632905
270,31.187223,694.981886,45.211240,5.662720,5.912701,5.632905
860,30.510552,776.447446,83.338518,5.662720,6.100385,5.895647
435,29.019383,29.973590,59.474095,5.662720,5.810522,5.895647


In [18]:
# 1. Min-Max Scaler with Leave-One-Out Encoder
scaler_mm_looe = MinMaxScaler()
numeric_columns_looe = ["temperature", "rainfall", "humidity", "soil_type", "weather_condition", "crop_type"]
X_train_looe_scaled = scaler_mm_looe.fit_transform(X_train_looe[numeric_columns_looe])
X_test_looe_scaled = scaler_mm_looe.transform(X_test_looe[numeric_columns_looe])

In [19]:
scaler_mm = MinMaxScaler()
X_train_ohe_scaled_mm = scaler_mm.fit_transform(X_train_ohe[["temperature", "rainfall", "humidity"]])
X_test_ohe_scaled_mm = scaler_mm.transform(X_test_ohe[["temperature", "rainfall", "humidity"]])

In [20]:
X_train_ohe_scaled_mm

array([[0.04202491, 0.8288506 , 0.65666461],
       [0.94477667, 0.11912103, 0.83772755],
       [0.60898076, 0.06520211, 0.52070157],
       ...,
       [0.77470275, 0.7761822 , 0.87055145],
       [0.69977605, 0.02685751, 0.39036443],
       [0.31125357, 0.88393057, 0.1574313 ]])

In [21]:
# Convert back to DataFrame for consistency
X_train_looe_scaled = pd.DataFrame(X_train_looe_scaled, columns=numeric_columns_looe, index=X_train.index)
X_test_looe_scaled = pd.DataFrame(X_test_looe_scaled, columns=numeric_columns_looe, index=X_test.index)

In [22]:
# Save the MinMaxScaler for LOOE
with open("scaler_mm_looe.pkl", "wb") as f:
    pickle.dump(scaler_mm_looe, f)

In [23]:
# 2. Feature Engineering: Add interaction terms and polynomial features
X_train_looe_scaled["temp_rainfall_interaction"] = X_train_looe_scaled["temperature"] * X_train_looe_scaled["rainfall"]
X_test_looe_scaled["temp_rainfall_interaction"] = X_test_looe_scaled["temperature"] * X_test_looe_scaled["rainfall"]
X_train_looe_scaled["temperature_squared"] = X_train_looe_scaled["temperature"] ** 2
X_test_looe_scaled["temperature_squared"] = X_test_looe_scaled["temperature"] ** 2

In [24]:
# 3. Random Forest Regression
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_looe_scaled, y_train)

RandomForestRegressor(random_state=42)

In [25]:
# Predict and evaluate
y_pred_train = rf_model.predict(X_train_looe_scaled)
y_pred_test = rf_model.predict(X_test_looe_scaled)

In [26]:
y_pred_train

array([6.99746033, 3.58755096, 7.10554965, 3.93115571, 7.2939603 ,
       7.18452251, 4.15807313, 4.043408  , 7.07733893, 4.5435014 ,
       4.50354773, 6.11490908, 5.02212959, 6.62321438, 8.37683513,
       4.21991158, 7.13854918, 4.15574749, 7.01647055, 4.66876783,
       4.97409987, 4.31724559, 6.28662335, 6.53777578, 5.68056115,
       7.50878958, 4.10191409, 8.34752391, 5.28467853, 6.47007601,
       7.50106072, 8.00039676, 6.86226904, 8.06573423, 4.69403539,
       8.19408041, 4.54436457, 8.37596637, 6.82548497, 5.60091769,
       7.20117018, 6.71894451, 8.34371791, 3.46574815, 5.48502669,
       7.25577115, 6.00606068, 6.99492117, 6.0140056 , 7.20542315,
       5.61958067, 4.21036649, 7.19567591, 7.51654799, 5.87023299,
       7.23807774, 6.96465103, 6.20184246, 6.03753836, 7.6319892 ,
       4.25872513, 6.65640735, 7.23549795, 4.10265716, 6.169842  ,
       4.60598229, 5.86626566, 4.72950583, 6.00227773, 5.38457747,
       4.78801517, 8.26376607, 8.34439811, 7.24438905, 4.91659

In [27]:
# Calculate metrics
train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

In [28]:
train_r2

0.85482953266945

In [29]:
print("Random Forest Regression Results:")
print(f"Train MSE: {train_mse:.4f}, Train R2: {train_r2:.4f}")
print(f"Test MSE: {test_mse:.4f}, Test R2: {test_r2:.4f}")

Random Forest Regression Results:
Train MSE: 0.8217, Train R2: 0.8548
Test MSE: 5.0021, Test R2: 0.0111


In [30]:

# Save the Random Forest model
with open("rf_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)

In [31]:
try:
    with open("looe_encoder.pkl", "rb") as f:
        loaded_looe_encoder = pickle.load(f)
    print("\nContents of looe_encoder.pkl:")
    print(f"Encoder object: {loaded_looe_encoder}")
    print(f"Columns encoded: {loaded_looe_encoder.cols}")
except FileNotFoundError:
    print("Error: looe_encoder.pkl not found.")

# View scaler_mm_looe.pkl
try:
    with open("scaler_mm_looe.pkl", "rb") as f:
        loaded_scaler_mm_looe = pickle.load(f)
    print("\nContents of scaler_mm_looe.pkl:")
    print(f"Feature ranges: {loaded_scaler_mm_looe.feature_range}")
    print(f"Min values: {loaded_scaler_mm_looe.data_min_}")
    print(f"Max values: {loaded_scaler_mm_looe.data_max_}")
except FileNotFoundError:
    print("Error: scaler_mm_looe.pkl not found.")

# View rf_model.pkl
try:
    with open("rf_model.pkl", "rb") as f:
        loaded_rf_model = pickle.load(f)
    print("\nContents of rf_model.pkl:")
    print(f"Model parameters: {loaded_rf_model.get_params()}")
    print(f"Feature importances: {dict(zip(X_train_looe_scaled.columns, loaded_rf_model.feature_importances_))}")
except FileNotFoundError:
    print("Error: rf_model.pkl not found.")


Contents of looe_encoder.pkl:
Encoder object: LeaveOneOutEncoder(cols=['soil_type', 'weather_condition', 'crop_type'])
Columns encoded: ['soil_type', 'weather_condition', 'crop_type']

Contents of scaler_mm_looe.pkl:
Feature ranges: (0, 1)
Min values: [15.09264046  3.2182636  40.0736911   5.66272016  5.77742025  5.6329055 ]
Max values: [ 34.99435347 999.41372577  89.77187581   6.00690217   6.10038493
   6.1539611 ]

Contents of rf_model.pkl:
Model parameters: {'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': None, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
Feature importances: {'temperature': np.float64(0.10756719908884614), 'rainfall': np.float64(0.17538644276468013), 'humidity': np.float64

In [49]:
train_r2

0.439939579255609

In [51]:
# View pickled objects (optional)
try:
    with open("looe_encoder.pkl", "rb") as f:
        loaded_looe_encoder = pickle.load(f)
    print("\nContents of looe_encoder.pkl:")
    print(f"Encoder object: {loaded_looe_encoder}")
    print(f"Columns encoded: {loaded_looe_encoder.cols}")
except FileNotFoundError:
    print("Error: looe_encoder.pkl not found.")

try:
    with open("scaler_mm_looe.pkl", "rb") as f:
        loaded_scaler_mm_looe = pickle.load(f)
    print("\nContents of scaler_mm_looe.pkl:")
    print(f"Feature ranges: {loaded_scaler_mm_looe.feature_range}")
    print(f"Min values: {loaded_scaler_mm_looe.data_min_}")
    print(f"Max values: {loaded_scaler_mm_looe.data_max_}")
except FileNotFoundError:
    print("Error: scaler_mm_looe.pkl not found.")

try:
    with open("rf_model.pkl", "rb") as f:
        loaded_rf_model = pickle.load(f)
    print("\nContents of rf_model.pkl:")
    print(f"Model parameters: {loaded_rf_model.get_params()}")
    print(f"Feature importances: {dict(zip(X_train_looe_scaled_df.columns, loaded_rf_model.feature_importances_))}")
except FileNotFoundError:
    print("Error: rf_model.pkl not found.")


Contents of looe_encoder.pkl:
Encoder object: LeaveOneOutEncoder(cols=['soil_type', 'weather_condition', 'crop_type'])
Columns encoded: ['soil_type', 'weather_condition', 'crop_type']

Contents of scaler_mm_looe.pkl:
Feature ranges: (0, 1)
Min values: [15.09264046  3.2182636  40.0736911   5.66272016  5.77742025  5.6329055 ]
Max values: [ 34.99435347 999.41372577  89.77187581   6.00690217   6.10038493
   6.1539611 ]

Contents of rf_model.pkl:
Model parameters: {'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 10, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 4, 'min_samples_split': 5, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
Feature importances: {'temperature': np.float64(0.28719733199009223), 'rainfall': np.float64(0.23277473737152787), 'humidity': np.float64(0

In [32]:
# PIPLINE Creaction 

In [33]:
sample = pd.DataFrame([{
    "temperature":30,
    "rainfall": 550,
    "humidity": 75,
    "soil_type": "Sandy",
    "weather_condition": "Sunny",
    "crop_type": "Corn"
}])

In [34]:
# PIPLINE Creaction 

In [35]:
sample = pd.DataFrame([{
    "temperature":30,
    "rainfall": 550,
    "humidity": 75,
    "soil_type": "Sandy",
    "weather_condition": "Sunny",
    "crop_type": "Corn"
}])

In [36]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

In [46]:
categorical_features = ["soil_type","weather_condition","crop_type"]
numeric_features = ["temperature","rainfall","humidity"]

In [47]:


preprocessor = ColumnTransformer(
    transformers=[
        ('encode', OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), categorical_features),
        ('scale', MinMaxScaler(), numeric_features)
    ]
)


In [39]:

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators = 100, random_state=42))
])


In [53]:

pipeline.fit(X_train,y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('encode',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['soil_type',
                                                   'weather_condition',
                                                   'crop_type']),
                                                 ('scale', MinMaxScaler(),
                                                  ['temperature', 'rainfall',
                                                   'humidity'])])),
                ('model',
                 RandomForestRegressor(max_depth=10, min_samples_leaf=4,
                                       min_samples_split=5, random_state=42))])

In [54]:
prediction = pipeline.predict(sample)
print(prediction[0])


5.1790003668017395


In [56]:
y_pred_train = pipeline.predict(X_train)
y_pred_test = pipeline.predict(X_test)
train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)


In [57]:
print("Random Forest Regression Results:")
print(f"Train MSE: {train_mse:.4f}, Train R2: {train_r2:.4f}")
print(f"Test MSE: {test_mse:.4f}, Test R2: {test_r2:.4f}")

Random Forest Regression Results:
Train MSE: 3.3356, Train R2: 0.4107
Test MSE: 5.0844, Test R2: -0.0052


In [58]:
# Save pipeline
with open("crop_yield_pipeline.pkl", "wb") as f:
    pickle.dump(pipeline, f)